## Data Collections

In [ ]:
import kaggle
import pandas as pd
import os
import csv

In [ ]:
# Télécharger le dataset depuis Kaggle
# !kaggle datasets download -d rishabhkausish/reddit-depression-dataset -p ../data/ --unzip

In [ ]:
# Chercher le fichier CSV extrait
csv_files = [f for f in os.listdir("../data/") if f.endswith(".csv")]
if csv_files:
    df = pd.read_csv(os.path.join("../data/", csv_files[0]))
    print("Dataset chargé dans le DataFrame 'df' est:", csv_files[0])
else:
    print("Aucun fichier CSV trouvé dans le dossier ../data/.")

Nettoyage des colonnes "Unnamed: 0" et "title" pour corriger les erreurs d'importation

In [ ]:
# Supprimer les lignes vides de la colonne Unnamed: 0
print(
    "Nombre de valeurs manquantes dans la colonne 'Unnamed: 0' avant suppression :",
    df["Unnamed: 0"].isna().sum(),
)
df = df.dropna(subset=["Unnamed: 0"])
print(
    "Nombre de valeurs manquantes dans la colonne 'Unnamed: 0' après suppression :",
    df["Unnamed: 0"].isna().sum(),
)

In [ ]:
# Supprimer les lignes vides de la colonne title
print(
    "Nombre de valeurs manquantes dans la colonne 'title' avant suppression :",
    df["title"].isna().sum(),
)
df = df.dropna(subset=["title"])
print(
    "Nombre de valeurs manquantes dans la colonne 'title' après suppression :",
    df["title"].isna().sum(),
)

In [ ]:
# Copie de sécurité
df_backup = df.copy(deep=True)

# Lignes où title contient au moins un chiffre
# mask = df["title"].astype(str).str.contains(r"\d", na=False) # Vérifie si la colonne "title" contient au moins un chiffre
unnamed_is_numeric = pd.to_numeric(
    df["Unnamed: 0"], errors="coerce"
).notna()  # Vérifie si la colonne "Unnamed: 0" est numérique
title_is_numeric = pd.to_numeric(
    df["title"], errors="coerce"
).notna()  # Vérifie si la colonne "title" est numérique
mask = (
    (~unnamed_is_numeric) & title_is_numeric
)  # On veut les lignes où "Unnamed: 0" n'est pas numérique et "title" est numérique

# Inversion des valeurs entre les 2 colonnes sur ces lignes
df.loc[mask, ["Unnamed: 0", "title"]] = df.loc[mask, ["title", "Unnamed: 0"]].to_numpy()

Pipeline de préparation pour NLP :
1) Vérifier que Unnamed: 0 est bien numérique partout
2) Vérifier que title est bien majoritairement non numérique
3) Inversion des valeurs entre les 2 colonnes sur ces lignes
4) Rename the column "Unnamed: 0" to "id"
5) Set the "id" column as the index of the DataFrame

In [ ]:
# Vérifier que Unnamed: 0 est bien numérique partout
print(
    f"Nombre de valeurs non numériques dans la colonne 'Unnamed: 0' : {pd.to_numeric(df['Unnamed: 0'], errors='coerce').isna().sum()}"
)

# Vérifier que title est bien majoritairement non numérique
print(
    f"Nombre de titres numériques dans la colonne 'title' : {pd.to_numeric(df['title'], errors='coerce').notna().sum()}"
)

# Si tout est ok, convertir définitivement Unnamed: 0 en int
# df["Unnamed: 0"] = df["Unnamed: 0"].astype(int)
df["Unnamed: 0"] = pd.to_numeric(df["Unnamed: 0"], errors="coerce").astype("Int64")

# Rename the column "Unnamed: 0" to "id"
df = df.rename(columns={"Unnamed: 0": "id"})

# Set the "id" column as the index of the DataFrame
df = df.set_index("id")

In [ ]:
# Supprime les lignes vides de la colonne "label"
print(
    f"Nombre de valeurs manquantes dans 'label' avant suppression : {df['label'].isna().sum()}"
)  # Affiche le nombre de valeurs manquantes dans la colonne "label" avant suppression
df = df.dropna(subset=["label"])
print(
    f"Nombre de valeurs manquantes dans 'label' après suppression : {df['label'].isna().sum()}"
)  # Affiche le nombre de valeurs manquantes dans la colonne "label" après suppression

In [ ]:
df.head()

In [ ]:
# Enregistrer le DataFrame nettoyé dans un nouveau fichier CSV
df.to_csv(
    "../data/reddit_depression_dataset_cleaned_v1.csv",
    index=False,
    quoting=csv.QUOTE_ALL,
)

In [ ]:
# Ouvre le CSV ../data/reddit_depression_dataset_cleaned_v1.csv
df = pd.read_csv("../data/reddit_depression_dataset_cleaned_v1.csv")

Préparation de la colonne clean_text :
1) Convertir les textes en minuscules
2) Supprimer les ponctuations
3) Supprimer les stop words
4) Appliquer le stemming ou le lemmatization

In [ ]:
# Distribution des labels
print(f"Distribution des labels :")
print(df["label"].value_counts(normalize=True) * 100)

In [ ]:
# Longueur des textes
df.loc[:, "text_length"] = df["title"].str.len() + df["body"].str.len()
print("Longueur textes :\n", df["text_length"].describe())

In [ ]:
# NaN restants
print(f"Nombre de valeurs manquantes dans le DataFrame : {df.isna().sum().sum()}")

In [ ]:
# Concat title + body (standard pour Reddit)
title_s = df["title"].fillna("").astype("string[python]")
body_s = df["body"].fillna("").astype("string[python]")
df["clean_text"] = title_s.str.cat(body_s, sep=" ")

# Nettoyage basique (vectorise, evite apply et limite le pic memoire)
df["clean_text"] = (
    df["clean_text"]
    .astype("string[python]")
    .str.replace(r"http\S+", "", regex=True)
    .str.replace(r"[^a-zA-Z\s]", "", regex=True)
    .str.lower()
    .str.strip()
    .astype("string[python]")
)

In [ ]:
# Drop "title" and "body" columns
df = df.drop(columns=["title", "body"])

In [ ]:
df.head()

In [ ]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv(
    "../data/reddit_depression_dataset_cleaned_v2.csv",
    index=False,
    quoting=csv.QUOTE_ALL,
)  # Use quoting=csv.QUOTE_ALL to ensure all fields are quoted, which can help with text containing commas or newlines
# indiquer à pandas de ne pas traiter les guillemets comme des caractères spéciaux. Cela résout souvent les erreurs de type 'EOF inside string' lorsque les données contiennent des guillemets non échappés. J'ajoute également encoding='utf-8' par bonne pratique.
# quoting=csv.QUOTE_NONE, encoding='utf-8', engine='python', on_bad_lines='skip') moteur de parsing 'python' de pandas, qui est plus robuste pour les fichiers mal formé
print(
    "✅ Dataset nettoyé sauvegardé !\nConcat title + body et nettoyage basique effectué dans la colonne 'clean_text'."
)

In [ ]:
# 4. Sauvegarde
# df[["label", "clean_text"]].to_parquet("reddit_depression_clean.parquet")
# print("✅ Dataset nettoyé sauvegardé !")

In [ ]:
print(f"Distribution des labels :")
print(df["label"].value_counts())

In [ ]:
df.shape

# Dataset prêt avec ~2M posts non-dépressifs (80,8%) et ~480k dépressifs (19,2%).